### ColumnTransformer in Scikit-learn

Real-world datasets often contain different column types that need different preprocessing. For example, a dataset may contain:

- Numerical columns with missing values.
- Ordinal categorical columns with a meaningful order.
- Nominal categorical columns with no natural order.
- Numerical columns that can be passed through without transformation.

Scikit-learn's **`ColumnTransformer`** lets you apply a separate transformer to each selected group of columns and combine the results into one feature matrix.

This notebook uses a COVID toy dataset and compares two approaches:

1. **Manual preprocessing:** transform each column group separately and combine the arrays.
2. **`ColumnTransformer`:** define the transformations once and let Scikit-learn apply them to the selected columns.

The target column is `has_covid`; it is kept separate from the input features.

In [1]:
import numpy as np
import pandas as pd 

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [3]:
df = pd.read_csv("Datasets/covid_toy.csv")
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [4]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [5]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [6]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [7]:
# train test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['has_covid']), df['has_covid'], test_size=0.2)

In [8]:
X_train

,age,gender,fever,cough,city
12,25,Female,99.0,Strong,Kolkata
44,20,Male,102.0,Strong,Delhi
41,82,Male,NaN,Mild,Kolkata
77,8,Female,101.0,Mild,Kolkata
6,14,Male,101.0,Strong,Bangalore
...,...,...,...,...,...
68,54,Female,104.0,Strong,Kolkata
5,84,Female,NaN,Mild,Bangalore
32,34,Female,101.0,Strong,Delhi
25,23,Male,NaN,Mild,Mumbai


In [9]:
y_train

12     No
44     No
41    Yes
77     No
6      No
     ... 
68     No
5     Yes
32    Yes
25     No
73    Yes
Name: has_covid, Length: 80, dtype: object

## Approach 1: Manual preprocessing

In this approach, each column group is transformed separately. Finally, the resulting NumPy arrays are concatenated horizontally.

#### Impute missing values in `fever`

`SimpleImputer()` uses the mean strategy by default for numerical data. `fit_transform()` learns the imputation value from the training column and fills missing values in that column.

For the test set, use `transform()` rather than `fit_transform()`. This applies the mean learned from the training set and prevents the test data from influencing preprocessing.

In [10]:
# adding simple imputer to fever columns 
si = SimpleImputer()

X_train_fever = si.fit_transform(X_train[['fever']])

X_test_fever = si.transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

#### Ordinal-encode `cough`

`OrdinalEncoder` maps the categories according to the specified order:

- `Mild` → 0
- `Strong` → 1

The order is explicitly supplied because the feature is treated as ordinal. Fit the encoder on `X_train`, then call `transform()` on `X_test` to preserve the same mapping.

In [11]:
# Ordinalencoding --> cough

oe = OrdinalEncoder(categories=[['Mild', 'Strong']])

X_train_cough = oe.fit_transform(X_train[['cough']])

X_test_cough = oe.transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

#### One-hot encode `gender` and `city`

`OneHotEncoder` creates binary columns for the nominal categories in `gender` and `city`.

- `sparse_output=False` requests a dense NumPy array.
- The encoder is fitted on the training columns.
- The fitted encoder transforms the test columns using the same learned categories.

For datasets with many categories, the default sparse output can be more memory-efficient. If new categories may appear at prediction time, consider `handle_unknown='ignore'`.

In [12]:
# OneHotEncoding --> gender, city 
ohe = OneHotEncoder(drop='first', sparse_output=False)

X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])

X_test_gender_city = ohe.transform(X_test[['gender', 'city']])

X_train_gender_city.shape

(80, 4)

In [13]:
# Extacitng age 
X_train_age = X_train.drop(columns=['gender', 'fever', 'cough', 'city']).values
X_test_age = X_test.drop(columns=['gender', 'fever', 'cough', 'city']).values
X_train_age.shape

(80, 1)

In [14]:
X_train_transformed = np.concatenate((X_train_age, X_train_fever, X_train_gender_city, X_train_cough), axis=1)
X_test_transformed = np.concatenate((X_test_age, X_test_fever, X_test_gender_city, X_test_cough), axis=1)

X_train_transformed

array([[ 25.,  99.,   0.,   0.,   1.,   0.,   1.],
       [ 20., 102.,   1.,   1.,   0.,   0.,   1.],
       [ 82., 101.,   1.,   0.,   1.,   0.,   0.],
       [  8., 101.,   0.,   0.,   1.,   0.,   0.],
       [ 14., 101.,   1.,   0.,   0.,   0.,   1.],
       [ 50., 103.,   0.,   0.,   1.,   0.,   0.],
       [ 16., 103.,   0.,   0.,   0.,   0.,   0.],
       [ 80.,  98.,   0.,   1.,   0.,   0.,   0.],
       [ 56., 104.,   0.,   0.,   0.,   0.,   1.],
       [ 26.,  98.,   0.,   0.,   1.,   0.,   0.],
       [ 80., 100.,   1.,   0.,   0.,   0.,   0.],
       [ 81., 101.,   0.,   0.,   0.,   1.,   0.],
       [ 27., 100.,   1.,   0.,   1.,   0.,   0.],
       [ 60.,  99.,   0.,   0.,   0.,   1.,   0.],
       [ 46., 103.,   1.,   0.,   0.,   0.,   1.],
       [ 14.,  99.,   0.,   0.,   0.,   1.,   0.],
       [ 33., 102.,   0.,   1.,   0.,   0.,   1.],
       [  5.,  98.,   0.,   0.,   0.,   1.,   1.],
       [  5., 102.,   1.,   0.,   1.,   0.,   0.],
       [ 55., 100.,   1.,   0.,

## Approach 2: Using `ColumnTransformer`

The manual approach works, but it requires several separate transformations and array-combination steps. `ColumnTransformer` makes this workflow more compact and easier to maintain.

#### Define the column transformations

The `transformers` argument is a list of tuples. Each tuple contains:

1. A name for the transformation.
2. The transformer object.
3. The column or columns to which it should be applied.

In this example:

| Transformer name | Transformation | Columns |
|---|---|---|
| `tnf1` | `SimpleImputer()` | `fever` |
| `tnf2` | `OrdinalEncoder(...)` | `cough` |
| `tnf3` | `OneHotEncoder()` | `gender`, `city` |

`remainder='passthrough'` keeps all columns not explicitly selected for transformation, such as `age`.

By default, `ColumnTransformer` concatenates the outputs in the order the transformers are listed, followed by the remainder columns.

In [15]:
from sklearn.compose import ColumnTransformer

In [16]:
transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(), ['fever']), 
    ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']), 
    ('tnf3', OneHotEncoder(), ['gender', 'city'])
], remainder='passthrough')

In [17]:
transformer.fit_transform(X_train)

array([[ 99.,   1.,   1.,   0.,   0.,   0.,   1.,   0.,  25.],
       [102.,   1.,   0.,   1.,   0.,   1.,   0.,   0.,  20.],
       [101.,   0.,   0.,   1.,   0.,   0.,   1.,   0.,  82.],
       [101.,   0.,   1.,   0.,   0.,   0.,   1.,   0.,   8.],
       [101.,   1.,   0.,   1.,   1.,   0.,   0.,   0.,  14.],
       [103.,   0.,   1.,   0.,   0.,   0.,   1.,   0.,  50.],
       [103.,   0.,   1.,   0.,   1.,   0.,   0.,   0.,  16.],
       [ 98.,   0.,   1.,   0.,   0.,   1.,   0.,   0.,  80.],
       [104.,   1.,   1.,   0.,   1.,   0.,   0.,   0.,  56.],
       [ 98.,   0.,   1.,   0.,   0.,   0.,   1.,   0.,  26.],
       [100.,   0.,   0.,   1.,   1.,   0.,   0.,   0.,  80.],
       [101.,   0.,   1.,   0.,   0.,   0.,   0.,   1.,  81.],
       [100.,   0.,   0.,   1.,   0.,   0.,   1.,   0.,  27.],
       [ 99.,   0.,   1.,   0.,   0.,   0.,   0.,   1.,  60.],
       [103.,   1.,   0.,   1.,   1.,   0.,   0.,   0.,  46.],
       [ 99.,   0.,   1.,   0.,   0.,   0.,   0.,   1.,

In [18]:
transformer.transform(X_test)

array([[ 99.,   1.,   1.,   0.,   0.,   1.,   0.,   0.,  59.],
       [101.,   0.,   1.,   0.,   0.,   1.,   0.,   0.,  75.],
       [100.,   0.,   0.,   1.,   1.,   0.,   0.,   0.,  11.],
       [101.,   1.,   0.,   1.,   0.,   0.,   1.,   0.,  71.],
       [104.,   0.,   0.,   1.,   0.,   0.,   0.,   1.,  44.],
       [103.,   0.,   0.,   1.,   0.,   0.,   1.,   0.,  83.],
       [ 98.,   1.,   0.,   1.,   0.,   0.,   0.,   1.,  23.],
       [100.,   0.,   1.,   0.,   0.,   0.,   1.,   0.,  19.],
       [100.,   0.,   0.,   1.,   1.,   0.,   0.,   0.,  10.],
       [100.,   0.,   0.,   1.,   0.,   1.,   0.,   0.,  27.],
       [101.,   0.,   0.,   1.,   0.,   1.,   0.,   0.,  42.],
       [103.,   0.,   1.,   0.,   0.,   0.,   1.,   0.,  69.],
       [ 98.,   0.,   1.,   0.,   1.,   0.,   0.,   0.,  64.],
       [ 98.,   0.,   0.,   1.,   1.,   0.,   0.,   0.,  73.],
       [102.,   1.,   1.,   0.,   1.,   0.,   0.,   0.,  82.],
       [ 98.,   1.,   1.,   0.,   0.,   0.,   1.,   0.,

### Advantages of `ColumnTransformer`

- Applies different preprocessing steps to different columns.
- Reduces repetitive code and manual array concatenation.
- Helps keep training and test transformations consistent.
- Works naturally with Scikit-learn `Pipeline`.
- Supports `remainder='passthrough'` or `remainder='drop'` for unselected columns.

#### Important notes and common mistakes

1. **Fit only on training data.** Use `fit_transform(X_train)` and then `transform(X_test)`.
2. **Do not fit a transformer separately on the test set.** This can produce inconsistent category mappings and leak information from the test set into preprocessing.
3. **Check the column selections.** A misspelled or missing column name causes an error.
4. **Know the output format.** Depending on the transformers and settings, the result may be a NumPy array or a sparse matrix.
5. **Remember the output order.** The output follows the order of the transformers, then the remainder columns when passed through.
6. **Consider unseen categories.** Configure `OneHotEncoder(handle_unknown='ignore')` if categories absent during fitting may appear later.
7. **Use a Pipeline for a complete ML workflow.** A `Pipeline` can chain the `ColumnTransformer` with a model so preprocessing is applied consistently during training and prediction.

### Key takeaways

- Use `ColumnTransformer` when different columns require different preprocessing.
- In this notebook, `fever` is imputed, `cough` is ordinal-encoded, `gender` and `city` are one-hot encoded, and `age` is passed through.
- Fit preprocessing on training data and reuse it to transform test data.
- `ColumnTransformer` is generally easier to maintain than manually transforming and concatenating many columns.